# +/− 2 trailing sigma：spread 的动态参考带

本课默认你已经有固定 beta 的配对价差：$S_t=Y_t-\beta X_t$。我们只讨论 spread 的 trailing mean、trailing sigma 与上下带；beta 在本课中固定。

建议顺序：先完成 `trailing_sigma_workshop.nb`，再阅读本教案。

## 1. trailing mean 的精确定义

设窗口长度为 $w$ 个交易日。在第 $t$ 日收盘后：

$$\mu_t=\frac{1}{w}\sum_{k=0}^{w-1}S_{t-k}$$

窗口严格是 $[t-w+1,t]$。$\mu_t$ 的第一个值出现在第 $w$ 个 spread 点，并应贴在该窗口的**最后一天**。这正是 `MovingAverage[spread, w]` 的输出对齐方式。

## 2. trailing sigma 的精确定义

对同一个窗口，样本标准差是：

$$\sigma_t=\sqrt{\frac{1}{w-1}\sum_{k=0}^{w-1}(S_{t-k}-\mu_t)^2}$$

Wolfram Language 的 `StandardDeviation` 对列表采用这个样本标准差约定。每个 $\sigma_t$ 都必须由一个独立的 trailing window 计算；整段历史只求一次标准差不是 rolling sigma。

## 3. +/− 2 sigma bands

上、下带定义为：

$$U_t=\mu_t+2\sigma_t,\qquad L_t=\mu_t-2\sigma_t$$

图中应该有四类对象：

- 蓝线：每一天的完整 spread $S_t$；
- 白线：从第 $w$ 日开始的 trailing mean；
- 红线：从同一第 $w$ 日开始的 $U_t,L_t$；
- 所有白线和红线的日期都应使用 `dates[[w ;;]]`。

## 4. 参考实现：列表怎样对齐

```wl
window = 20;
mean = MovingAverage[spread, window];
sigma = StandardDeviation /@ Partition[spread, window, 1];
bandDates = dates[[window ;;]];
upper = mean + 2 sigma;
lower = mean - 2 sigma;

{Length[mean], Length[sigma], Length[bandDates], Length[upper], Length[lower]}
```

最后一行应给出五个相同的长度。`Partition[spread, window, 1]` 的每个子列表都恰好是一个从旧到新的 trailing window。

## 5. 无未来函数到底到哪一步？

在 $t$ 日收盘后，$S_t,\mu_t,\sigma_t,U_t,L_t$ 都可计算，因为它们没有用到 $S_{t+1}$。这是**计算无未来函数**。

但交易时点要再区分：

- 若信号在 $t$ 日收盘后生成、下一根 bar 执行，可使用当日的 $\mu_t,\sigma_t$。
- 若必须在 $t$ 日开盘前决定，则只能使用 $\mu_{t-1},\sigma_{t-1}$。

同一公式是否有前视，取决于信息在何时可得和订单在何时执行。

## 6. 为什么不是“超过 ±2σ 就有 95% 把握”？

正态分布下约 95% 的经典说法，要求独立、同分布且参数已知等强假设。真实 spread 通常有自相关、肥尾、波动聚集、结构变化，而且 $\mu_t,\sigma_t$ 本身由重叠窗口估计。

所以 ±2 trailing sigma 是一个**相对偏离刻度**，不是反转概率承诺。它可用于提出交易假设，但仍需用冻结 beta、独立样本和明确执行规则检验。

## 7. 常见调试错误

| 现象 | 原因 | 修正 |
| --- | --- | --- |
| `MovingAverage` 报列表错误 | `spread` 还不是数值列表 | 先检查 `Head[spread]` 与 `Take[spread, 3]` |
| 图报长度不一致 | mean/sigma 和完整 dates 混用了 | bands 一律配 `dates[[window ;;]]` |
| 上下带只有一对直线 | 对全样本只算了一次 sigma | 对 `Partition[..., window, 1]` 的每个窗口求 sigma |
| 早期日期也有 band | 日期被错贴到了窗口第一天 | 第一个 band 应在第 `window` 个日期 |
| 结果看似很漂亮却无法实盘复现 | 用全样本 beta 或未来窗口 | 在训练期冻结 beta，并按可执行时点滞后信号 |

## 8. 回头看工作纸

完成工作纸后，重新回答三个问题：

1. 为什么 mean、sigma、upper、lower 的长度相同，却比 spread 短 $w-1$？
2. 为什么白线不是端点趋势线？
3. 如果 beta 改变，为什么白线和红带都会变化？

如果这三点都能自己解释，你已经掌握了本课真正的核心。